In [11]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import joblib
from sklearn.base import BaseEstimator, TransformerMixin


In [12]:
joblib.dump(scaler, "../models/customer_scaler.pkl")
joblib.dump(kmeans, "../models/customer_kmeans.pkl")

['../models/customer_kmeans.pkl']

In [13]:
import os

print(os.path.exists("../models/customer_scaler.pkl"))
print(os.path.exists("../models/customer_kmeans.pkl"))

True
True


In [14]:
df = pd.read_csv("../data/shopping_trends.csv")
X = pd.read_csv("../data/clustering_features.csv")
reference_labels = np.load("../data/reference_labels.npy")

In [15]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

labels = kmeans.fit_predict(X_scaled)
print("X_scaled:", X_scaled.shape)
print("Labels:", labels.shape)
print("Clusters:", np.unique(labels))

X_scaled: (3900, 8)
Labels: (3900,)
Clusters: [0 1]


In [16]:
class CustomerFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.frequency_map = {
            "Weekly": 52,
            "Bi-Weekly": 26,
            "Fortnightly": 26,
            "Monthly": 12,
            "Quarterly": 4,
            "Every 3 Months": 4,
            "Annually": 1
        }

        self.final_features = [
            "Age",
            "Purchase Amount (USD)",
            "Review Rating",
            "Previous Purchases",
            "Purchase Frequency",
            "Is_Subscribed",
            "Log_Annual_Spend",
            "Log_Purchase_Engagement"
        ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        features = pd.DataFrame({
            "Age": X["Age"],
            "Purchase Amount (USD)": X["Purchase Amount (USD)"],
            "Review Rating": X["Review Rating"],
            "Previous Purchases": X["Previous Purchases"],
            "Purchase Frequency": X["Frequency of Purchases"].map(
                self.frequency_map
            ),
            "Is_Subscribed": X["Subscription Status"].map({
                "Yes": 1,
                "No": 0
            })
        })

        features["Estimated_Annual_Spend"] = (
            features["Purchase Amount (USD)"]
            * features["Purchase Frequency"]
        )

        features["Purchase_Engagement"] = (
            features["Previous Purchases"]
            * features["Purchase Frequency"]
        )

        features["Log_Annual_Spend"] = np.log1p(
            features["Estimated_Annual_Spend"]
        )

        features["Log_Purchase_Engagement"] = np.log1p(
            features["Purchase_Engagement"]
        )

        return features[self.final_features]

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [18]:
customer_pipeline = Pipeline([
    ("features", CustomerFeatureTransformer()),
    ("scaler", StandardScaler()),
    ("model", KMeans(
        n_clusters=2,
        random_state=42,
        n_init=10
    ))
])

In [19]:
customer_pipeline.fit(df)

Pipeline(steps=[('features', CustomerFeatureTransformer()),
                ('scaler', StandardScaler()),
                ('model', KMeans(n_clusters=2, n_init=10, random_state=42))])

In [20]:
pipeline_labels = customer_pipeline.predict(df)

print("Shape:", pipeline_labels.shape)
print("Clusters:", np.unique(pipeline_labels))

Shape: (3900,)
Clusters: [0 1]


In [21]:
from sklearn.metrics import adjusted_rand_score, silhouette_score

ari = adjusted_rand_score(
    reference_labels,
    pipeline_labels
)

silhouette = silhouette_score(
    customer_pipeline.named_steps["scaler"].transform(
        customer_pipeline.named_steps["features"].transform(df)
    ),
    pipeline_labels
)

print("ARI:", ari)
print("Silhouette:", silhouette)

ARI: 1.0
Silhouette: 0.21232643227860026


In [22]:
new_customer = pd.DataFrame([{
    "Age": 35,
    "Purchase Amount (USD)": 72,
    "Review Rating": 4.0,
    "Previous Purchases": 30,
    "Frequency of Purchases": "Monthly",
    "Subscription Status": "Yes"
}])

In [23]:
predicted_cluster = customer_pipeline.predict(new_customer)

print("Predicted cluster:", predicted_cluster[0])

Predicted cluster: 1


In [24]:
centers_scaled = customer_pipeline.named_steps["model"].cluster_centers_

centers_original = customer_pipeline.named_steps["scaler"].inverse_transform(
    centers_scaled
)

centers_df = pd.DataFrame(
    centers_original,
    columns=customer_pipeline.named_steps["features"].final_features
)

centers_df

,Age,Purchase Amount (USD),Review Rating,Previous Purchases,Purchase Frequency,Is_Subscribed,Log_Annual_Spend,Log_Purchase_Engagement
0,44.207557,58.450378,3.745592,23.888161,4.362720,0.263476,5.122221,4.037609
1,43.924282,61.126371,3.754465,26.868407,31.059008,0.276762,7.355928,6.392435


# Sauvegarde

In [25]:
import joblib

joblib.dump(
    customer_pipeline,
    "../models/customer_pipeline.pkl"
)

['../models/customer_pipeline.pkl']

In [26]:
import os

print(os.path.exists("../models/customer_pipeline.pkl"))

True


In [27]:
loaded_pipeline = joblib.load(
    "../models/customer_pipeline.pkl"
)

In [28]:
loaded_prediction = loaded_pipeline.predict(new_customer)

print("Original prediction:", predicted_cluster[0])
print("Loaded prediction:", loaded_prediction[0])

Original prediction: 1
Loaded prediction: 1


# Decision Layer

In [29]:
def assign_customer_segment(customer):
    cluster = loaded_pipeline.predict(customer)[0]
    return cluster

In [30]:
cluster = assign_customer_segment(new_customer)

print("Assigned cluster:", cluster)

Assigned cluster: 1


In [31]:
cluster_profiles = centers_df.copy()

cluster_profiles.index.name = "Cluster"

cluster_profiles

,Age,Purchase Amount (USD),Review Rating,Previous Purchases,Purchase Frequency,Is_Subscribed,Log_Annual_Spend,Log_Purchase_Engagement
Cluster,,,,,,,,
0,44.207557,58.450378,3.745592,23.888161,4.362720,0.263476,5.122221,4.037609
1,43.924282,61.126371,3.754465,26.868407,31.059008,0.276762,7.355928,6.392435


In [36]:
def get_customer_segment_summary(customer):
    
    cluster = loaded_pipeline.predict(customer)[0]
    
 
    
    return {
        "Cluster": cluster,
        "Segment": interpret_cluster(cluster)["Segment"],
        "Main characteristic": interpret_cluster(cluster)["Main characteristic"],
        "Purchase Frequency": customer["Frequency of Purchases"].iloc[0],
        "Previous Purchases": customer["Previous Purchases"].iloc[0],
        "Purchase Amount": customer["Purchase Amount (USD)"].iloc[0],
        "Subscription": customer["Subscription Status"].iloc[0]
    }

In [37]:
summary = get_customer_segment_summary(new_customer)

summary

{'Cluster': 1,
 'Segment': 'Higher Purchase Intensity',
 'Main characteristic': 'Higher purchase frequency',
 'Purchase Frequency': 'Monthly',
 'Previous Purchases': 30,
 'Purchase Amount': 72,
 'Subscription': 'Yes'}

In [38]:
cluster = loaded_pipeline.predict(new_customer)[0]

cluster_profile = cluster_profiles.loc[cluster]

cluster_profile

Age                        43.924282
Purchase Amount (USD)      61.126371
Review Rating               3.754465
Previous Purchases         26.868407
Purchase Frequency         31.059008
Is_Subscribed               0.276762
Log_Annual_Spend            7.355928
Log_Purchase_Engagement     6.392435
Name: 1, dtype: float64

In [39]:
customer_frequency = (
    new_customer["Frequency of Purchases"]
    .map(loaded_pipeline.named_steps["features"].frequency_map)
    .iloc[0]
)

comparison = pd.DataFrame({
    "Customer": [
        customer_frequency,
        new_customer["Previous Purchases"].iloc[0],
        new_customer["Purchase Amount (USD)"].iloc[0]
    ],
    "Cluster Center": [
        cluster_profile["Purchase Frequency"],
        cluster_profile["Previous Purchases"],
        cluster_profile["Purchase Amount (USD)"]
    ]
}, index=[
    "Purchase Frequency (annualized)",
    "Previous Purchases",
    "Purchase Amount"
])

comparison

,Customer,Cluster Center
Purchase Frequency (annualized),12,31.059008
Previous Purchases,30,26.868407
Purchase Amount,72,61.126371


In [40]:
comparison["Difference (%)"] = (
    (comparison["Customer"] - comparison["Cluster Center"])
    / comparison["Cluster Center"]
) * 100

comparison

,Customer,Cluster Center,Difference (%)
Purchase Frequency (annualized),12,31.059008,-61.363866
Previous Purchases,30,26.868407,11.655297
Purchase Amount,72,61.126371,17.788770


In [41]:
segment_actions = pd.DataFrame({
    "Segment": [
        "Lower Purchase Intensity",
        "Higher Purchase Intensity"
    ],
    "Observed Pattern": [
        "Lower purchase frequency",
        "Higher purchase frequency"
    ],
    "Business Objective": [
        "Increase engagement and purchase frequency",
        "Maintain engagement and prioritize commercial attention"
    ],
    "Recommended Action": [
        "Test targeted re-engagement campaigns",
        "Test retention and cross-sell initiatives"
    ],
    "Measurement": [
        "Change in purchase frequency",
        "Repeat purchase rate and incremental revenue"
    ]
})

segment_actions

,Segment,Observed Pattern,Business Objective,Recommended Action,Measurement
0,Lower Purchase Intensity,Lower purchase frequency,Increase engagement and purchase frequency,Test targeted re-engagement campaigns,Change in purchase frequency
1,Higher Purchase Intensity,Higher purchase frequency,Maintain engagement and prioritize commercial ...,Test retention and cross-sell initiatives,Repeat purchase rate and incremental revenue


In [42]:
print("Pipeline:", type(loaded_pipeline).__name__)
print("Number of clusters:", loaded_pipeline.named_steps["model"].n_clusters)
print("Number of model features:", len(
    loaded_pipeline.named_steps["features"].final_features
))
print("Model features:", loaded_pipeline.named_steps["features"].final_features)

Pipeline: Pipeline
Number of clusters: 2
Number of model features: 8
Model features: ['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases', 'Purchase Frequency', 'Is_Subscribed', 'Log_Annual_Spend', 'Log_Purchase_Engagement']


In [22]:
import sys
sys.path.append("..")

from src.customer_features import CustomerFeatureTransformer

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
customer_pipeline = Pipeline([
    ("features", CustomerFeatureTransformer()),
    ("scaler", StandardScaler()),
    ("model", KMeans(
        n_clusters=2,
        random_state=42,
        n_init=10
    ))
])

In [23]:
customer_pipeline.fit(df)

Pipeline(steps=[('features', CustomerFeatureTransformer()),
                ('scaler', StandardScaler()),
                ('model', KMeans(n_clusters=2, n_init=10, random_state=42))])

In [25]:
pipeline_labels = customer_pipeline.predict(df)

print("Number of predictions:", len(pipeline_labels))
print("Clusters:", sorted(set(pipeline_labels)))

Number of predictions: 3900
Clusters: [0, 1]


In [18]:
import numpy as np

reference_labels = np.load("../data/reference_labels.npy")

print("Reference labels:", len(reference_labels))
print("Clusters:", sorted(set(reference_labels)))

Reference labels: 3900
Clusters: [0, 1]


In [26]:
pipeline_labels = customer_pipeline.predict(df)

print("Number of predictions:", len(pipeline_labels))
print("Clusters:", sorted(set(pipeline_labels)))

Number of predictions: 3900
Clusters: [0, 1]


In [27]:
from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(
    reference_labels,
    pipeline_labels
)

print("ARI:", ari)

ARI: 1.0


In [28]:
import joblib

joblib.dump(
    customer_pipeline,
    "../models/customer_pipeline.pkl"
)

print("Pipeline saved successfully.")

Pipeline saved successfully.


In [29]:
loaded_pipeline = joblib.load(
    "../models/customer_pipeline.pkl"
)

print(type(loaded_pipeline))

<class 'sklearn.pipeline.Pipeline'>


In [30]:
print(
    type(
        loaded_pipeline.named_steps["features"]
    )
)

print(
    loaded_pipeline.named_steps["features"].final_features
)

<class 'src.customer_features.CustomerFeatureTransformer'>
['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases', 'Purchase Frequency', 'Is_Subscribed', 'Log_Annual_Spend', 'Log_Purchase_Engagement']


In [32]:
import pandas as pd

new_customer = pd.DataFrame([{
    "Age": 35,
    "Purchase Amount (USD)": 72,
    "Review Rating": 4.0,
    "Previous Purchases": 30,
    "Frequency of Purchases": "Monthly",
    "Subscription Status": "Yes"
}])

In [33]:
loaded_prediction = loaded_pipeline.predict(new_customer)

print("Predicted cluster:", loaded_prediction[0])

Predicted cluster: 1
